# Option 1: Ensemble learning
The main objective of this assignment is to explore ensemble learning as a means of predicting cyberattacks.
Reusing the dataset provided for assignment 2, select an appropriate ensemble method and tune this method
accordingly. Ensure that the choice of ensemble method and the tuning performed are well justified. Any
necessary data transformations should also be performed where well justified. Evaluate the tuned ensemble
method on the dataset and compare the results with the models used in assignment 2.  
  
**You may select any ensemble learning model, but approach this decision with intention. What do you wish the
final evaluation to reflect? Do you want to compare ensemble methods that are similar to the models used in
assignment 2, or do you wish to want to compare dissimilar methods? Neither option is correct, but a sound
argument must be supplied.**  
  
The tuning of your chosen ensemble method should also be performed with an aim to answer specific questions
that are set out before evaluation. Not only do you want a model that performs well, but you also want to gain
insight from the tuning performance.  
  
You report should include the necessary background, detail on your chosen ensemble method and the implementation thereof, detail on the empirical process followed and then finally your results and a discussion of the
results. Remember that you must include all necessary background and implementation details for the models
used in assignment 2 that you compare against. You may revise or improve your assignment 2 implementation.

# Rewind: assignment 2
Assignment 2 compares decision trees and KNN for classification. The problem is Scikit-learn's implementation has fundamental limitations:  
1. Sci-kit learn implements CART, not C4.5 as described in the course material
2. Binary splits only - no multi-way categorical splits
3. No gain ratio -> criterion limited to Gini or entropy. Gain ratio, which is C4.5's splitting criterion and a way to combat the many-value bias, is not available
4. Different pruning mechanism. CART uses "minimal cost-complexity pruning (ccp_alpha)," which you note "differs from the bottom-up, validation-performance-driven pruning mentioned in the Background section" (the reduced-error-style pruning on slides 31–32).
5. Cannot handle missing values natively. You note "Missing values have to be resolved before fitting, since DecisionTreeClassifier cannot perform C4.5's process of splitting instances with missing values across all branches in proportion to branch frequency." (That fractional-weight procedure is precisely slides 35–38: weight $w_i(D_o) = w_i(D)p_o$, classify by the resulting class distribution.)
6. Requires purely numeric input -> forces one-hot encoding. In the data-quality/preprocessing discussion you note "scikit-learn requires all features to be purely numeric," which is why proto (133), state, service all had to be one-hot encoded - reintroducing a dimensionality problem that C4.5's native multi-way categorical handling would have avoided.  
  
## Recommended ensemble
- Want to use random forests


# NOTE:
- what is decorrelation of base learners?
- AdaBoost is sensitive to outliers/noise: it repeatedly increases the weight of misclassified instances. Outliers (and noisy points) are hard to classify correctly, so they keep getting misclassified and their weights keep growing — the ensemble then devotes later weak learners to chasing those points, distorting the decision boundary. (**This is why it is not appropriate for assignment 4**)
- CART implementation used **class_weight='balanced'** (inverse-frequency class weighting) which assigns a weight to each class inversely proportional to how frequently the class occurs. Rare classes (Worms, Analysis, Backdoor) count for more during split selection, leaf-size checks, and pruning, counteracting the 534:1 skew. The headline is misclassifying an observation with a rarer class incurs a higher penalty (higher impurity score)
- Things to understand: **CART**, **Entropy + Information gain**, **Gain ratio**, **CART Forest training + OOB**  
  
**Entropy:** $$H(S) = -\sum_{c=1}^{M} p_c \log p_c$$
- $p_c$ is the fraction of the observations at that node whose target class is $c$
- If $p_c = 0$ then its contribution is 0
- Pure node (all one class): one $p_c = 1$, rest $0$ → $H = 0$. No uncertainty.
- Perfectly mixed (all classes equally likely, $p_c = 1/M$): entropy is maximal. Maximum uncertainty.
- Anything in between sits between those extremes.  
  
**Information gain:** A split partitions the parent set $S$ into children $S_1, \dots, S_k$ (two for a numeric threshold, one per category for a C4.5 multi-way split). Information gain is the entropy of the parent minus the weighted average entropy of the children:  
$$\mathrm{IG}(S) = H(S) - \sum_{j=1}^{k} \frac{|S_j|}{|S|}, H(S_j)$$
So the weight a child node contributes to the summed entropy, is proportional to the number of observations in that child node.  
This introduces the concept of **Information gain's bias towards many-valued, high cardinality splits.** Information gain has a bias toward many-valued (high-cardinality) splits. The extreme case makes it obvious: imagine splitting on a near-unique feature like a row ID. Every branch gets one observation, every branch is trivially pure, so the weighted child entropy is ~0 and $\mathrm{IG}$ is maximal — yet the split is useless, it generalises to nothing. IG rewards a split for making children pure, and fragmenting the data into many tiny branches is a cheap way to look pure without being informative.  
  
**Split information:** Split information (a.k.a. intrinsic information) measures how much a split fragments the data, computed as the entropy of the branch sizes — not the class labels:

$$\mathrm{SI}(S) = -\sum_{j=1}^{k} \frac{|S_j|}{|S|} \log \frac{|S_j|}{|S|}$$

Note the structural parallel to entropy, but the $\tfrac{|S_j|}{|S|}$ are branch proportions (what fraction of instances go down branch $j$), whereas in $H(S)$ the $p_c$ were class proportions. So:
- A binary, balanced split (two branches of equal size): $\mathrm{SI} = 1$ bit — low.
- A many-way split (e.g. 133 proto categories): $\mathrm{SI} = \log k$ — large, grows with the number of branches.
- A split sending almost everything down one branch: $\mathrm{SI} \approx 0$ — it barely fragments.  
  
**Gain ratio:** $$\mathrm{GainRatio}(S) = \frac{\mathrm{IG}(S)}{\mathrm{SI}(S)}$$

You divide the gain by how much the split fragments the data. A feature now only wins if its gain is high relative to its branching. The ID-style split still has huge $\mathrm{IG}$, but its $\mathrm{SI} = \log N$ is enormous, so its gain ratio collapses — exactly the demotion you want. (**This is what C4.5 uses by default**)  
  
**How C4.5 single tree performs hyperparameter tuning:** Simply 5 fold cross validation.
  
  
**How C4.5 FOREST performs hyperparameter tuning** Scored using out-of-bag (OOB), not CV. In this setup 5-fold CV would mean fitting 5 forests per config — and a forest is hundreds of trees. Too expensive. So each config is scored by fitting one forest and reading its OOB macro-F1. That's the whole reason OOB exists in your pipeline: it gives a held-out-style estimate from a single fit, replacing the inner CV loop.  
**OOB from scratch:** The key fact is how each tree in a forest is trained: not on the full data, but on a bootstrap sample — draw $N$ observations with replacement from the $N$ training observations. Because of replacement, some observations are drawn several times and some aren't drawn at all.  

How many get left out? The chance a specific observation is missed in one draw is $(1 - \tfrac{1}{N})$; missed in all $N$ draws is $(1-\tfrac{1}{N})^N \to e^{-1} \approx 0.368$. So each tree trains on ~63.2% of the data and leaves ~36.8% unseen. Those unseen observations are that tree's out-of-bag set.  

Now the trick. Take one observation $x$. Across the whole forest, some trees had $x$ in their bootstrap sample ("in-bag") and some didn't ("out-of-bag for $x$"). To predict $x$ honestly, let only the trees that never trained on $x$ vote:  
  
Forest of 100 trees, observation $x$. Say $x$ happened to be in-bag for 63 trees and OOB for 37. Its OOB prediction is the majority vote of just those 37 trees — none of which ever saw $x$ during training.  
  
Do that for every observation → you get a prediction for (almost) every point, each made only by trees that didn't train on it. Score those OOB predictions against the true labels → OOB macro-F1.  
**NOTE:** OOB is noisier with few trees. Each OOB prediction uses only ~37% of the forest, so at small n_estimators fewer trees vote per point and the estimate wobbles; it stabilises as trees grow. (This is exactly why the plateau sweep we discussed is trustworthy only at large $B$.)
  
**What it means for base learners to be decorrelated and why it's important:**  
"Decorrelated" means the individual trees don't all make the same predictions and, crucially, the same mistakes. It's the single most important condition for a forest to beat a single tree. Here's the why, with the one formula that makes it click.  
  
Bagging works by averaging many trees to cut variance. But how much variance you kill depends on how correlated the trees are. If you average $B$ trees that are identically distributed, each with variance $\sigma^2$, and with pairwise correlation $\rho$ between them, the variance of the ensemble is:  
 
$$\mathrm{Var}_{\text{ensemble}} = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

Read the two terms:  

- $\frac{1-\rho}{B}\sigma^2$ — shrinks toward 0 as you add trees ($B \to \infty$). This is the part "more trees" fixes.
- $\rho,\sigma^2$ — a floor that adding trees cannot touch. It's set entirely by the correlation $\rho$.  
  
So no matter how many trees you throw in, your ensemble variance can't drop below $\rho\sigma^2$. The only way to lower that floor is to reduce $\rho$ — i.e. decorrelate the trees. That's the whole game.  
  
Averaging only cancels errors that are independent. If every tree makes the same mistake on observation $x$ (because they all learned the same structure), the majority vote repeats that mistake — the ensemble is no better than one tree. If the trees' mistakes are spread around different observations, they wash out in the vote and the aggregate is much more accurate.  

**How the random forest decorrelates:**  
- Bootstrap sampling already gives some decorrelation: each tree sees a different ~63% resample of the data.
- But that's not enough. If one feature is very predictive, every tree will split on it first and they'll all end up looking similar → high $\rho$.
- So random forests add feature subspace sampling: at each split only a random subset of ~$\sqrt{p}$ features is even considered. This forces different trees to build around different features, so they capture different structure and their errors stop lining up → lower $\rho$. This is the step that distinguishes a random forest from plain bagging.

  
**CART**: 

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import time
import subprocess
import pathlib

In [21]:
attack_category_map = {
    0: "Normal",
    1: "Reconnaissance",
    2: "Backdoor",
    3: "DoS",
    4: "Exploits",
    5: "Analysis",
    6: "Fuzzers",
    7: "Worms",
    8: "Shellcode",
    9: "Generic",
}

df = pd.read_csv("networkTraffic.csv", na_values="?")

df.shape

(257673, 44)

# Preprocessing — CART pipeline (reused from Assignment 2)

This reproduces the Assignment 2 classification-tree preprocessing so that the **CART single tree** and the **CART random forest** share an identical input pipeline. Because scikit-learn's CART requires purely numeric input, the categorical features are one-hot encoded: `service` (with its 54.8% missingness encoded as its own `"missing"` category, since the missingness is itself informative), `proto` (133 categories), `state`, and `ct_state_ttl`. The 26 invalid `is_ftp_login` rows (values 2, 4) are removed. All numeric features are kept raw, since trees are invariant to monotonic scaling.


In [22]:
df_preprocessed = pd.DataFrame(index=df.index)

#service - encoding

#Replace missing values with "missing" and perform one-hot encoding
service = df["service"].fillna("missing")
service_encoded = pd.get_dummies(service, prefix="service")
df_preprocessed = pd.concat([df_preprocessed, service_encoded], axis=1)

#proto - encoding
proto_encoded = pd.get_dummies(df["proto"], prefix="proto")
df_preprocessed = pd.concat([df_preprocessed, proto_encoded], axis=1)
#state - encoding
state_encoded = pd.get_dummies(df["state"], prefix="state")
df_preprocessed = pd.concat([df_preprocessed, state_encoded], axis=1)
#ct_state_ttl - encoding
ct_state_ttl_encoded = pd.get_dummies(df["ct_state_ttl"], prefix="ct_state_ttl")
df_preprocessed = pd.concat([df_preprocessed, ct_state_ttl_encoded], axis=1)

#Add the rest of the predictors
df_preprocessed["dur"] = df["dur"]
df_preprocessed["spkts"] = df["spkts"]
df_preprocessed["dpkts"] = df["dpkts"]
df_preprocessed["sbytes"] = df["sbytes"]
df_preprocessed["dbytes"] = df["dbytes"]
df_preprocessed["rate"] = df["rate"]
df_preprocessed["sttl"] = df["sttl"]
df_preprocessed["dttl"] = df["dttl"]
df_preprocessed["sload"] = df["sload"]
df_preprocessed["dload"] = df["dload"]
df_preprocessed["sloss"] = df["sloss"]
df_preprocessed["dloss"] = df["dloss"]
df_preprocessed["sinpkt"] = df["sinpkt"]
df_preprocessed["dinpkt"] = df["dinpkt"]
df_preprocessed["sjit"] = df["sjit"]
df_preprocessed["djit"] = df["djit"]
df_preprocessed["swin"] = df["swin"]
df_preprocessed["stcpb"] = df["stcpb"]
df_preprocessed["dtcpb"] = df["dtcpb"]
df_preprocessed["dwin"] = df["dwin"]
df_preprocessed["tcprtt"] = df["tcprtt"]
df_preprocessed["synack"] = df["synack"]
df_preprocessed["ackdat"] = df["ackdat"]
df_preprocessed["smean"] = df["smean"]
df_preprocessed["dmean"] = df["dmean"]
df_preprocessed["trans_depth"] = df["trans_depth"]
df_preprocessed["response_body_len"] = df["response_body_len"]
df_preprocessed["ct_srv_src"] = df["ct_srv_src"]
df_preprocessed["ct_dst_ltm"] = df["ct_dst_ltm"]
df_preprocessed["ct_src_dport_ltm"] = df["ct_src_dport_ltm"]
df_preprocessed["ct_dst_sport_ltm"] = df["ct_dst_sport_ltm"]
df_preprocessed["ct_dst_src_ltm"] = df["ct_dst_src_ltm"]
df_preprocessed["is_ftp_login"] = df["is_ftp_login"]
df_preprocessed["ct_ftp_cmd"] = df["ct_ftp_cmd"]
df_preprocessed["ct_flw_http_mthd"] = df["ct_flw_http_mthd"]
df_preprocessed["ct_src_ltm"] = df["ct_src_ltm"]
df_preprocessed["ct_srv_dst"] = df["ct_srv_dst"]
df_preprocessed["is_sm_ips_ports"] = df["is_sm_ips_ports"]

# remove observations where is_ftp_login has invalid values
is_ftp_login_valid = (df["is_ftp_login"] == 0) | (df["is_ftp_login"] == 1)
df_preprocessed = df_preprocessed.loc[is_ftp_login_valid]

df_preprocessed.shape

(257647, 202)

# CART tree - hyperparameter tuning
- **NOTE** the hyperparameters to choose over stay the same as they were for assignment 2, the headline is **gain ratio** is not available with *CART* and therefore cannot be added to the grid. **Gain ratio** is information gain normalised by split information, the normalization is exactly the remedy for the *many-value bias* that comes with high cardinality features. **However,** this is something that we can add to the tuning grid when we consider the C4.5 random forest. (Gain ratio isn't available with CART because the tree only allows binary splits anyway)
- Identical to Assignment 2: a stratified **50,000-observation tuning set** (for hyperparameter search) held separate from the **~207,647 train/test set** (for the final reported performance), using `random_state=42`. This same split is **shared across all four models**

In [5]:
X = df_preprocessed
y = df.loc[is_ftp_login_valid, "attack_cat"]

# stratify=y preserves the target's class distribution in both sets
X_tune, X_train_test, y_tune, y_train_test = train_test_split(
    X, y, train_size=50000, stratify=y, random_state=42
)

print("Tuning set:     ", X_tune.shape)
print("Train/test set: ", X_train_test.shape)
print("Classes present in tuning set:", sorted(y_tune.unique()))

Tuning set:      (50000, 202)
Train/test set:  (207647, 202)
Classes present in tuning set: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]


In [6]:
#Confirming the CART tree tuning from assignment 2

# generate a sensible ccp_alpha candidate range from the actual pruning path,
# rather than guessing arbitrary values
path = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_tune, y_tune)
ccp_alphas = path.ccp_alphas

# the full path can have thousands of points - subsample a manageable spread
alpha_grid = np.unique(np.quantile(ccp_alphas, np.linspace(0, 1, 8)))
alpha_grid = np.clip(alpha_grid, 0, None)

param_grid = {
    "criterion": ["gini", "entropy"],
    "min_samples_leaf": [1, 5, 15, 30],
    "class_weight": [None, "balanced"],
    "ccp_alpha": alpha_grid,
}

tree = DecisionTreeClassifier(random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=tree,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
)

start = time.time()
grid_search.fit(X_tune, y_tune)
print(f"grid search took {time.time() - start:.1f}s")

print("Best params:", grid_search.best_params_)
print(f"Best tuning macro-F1: {grid_search.best_score_:.4f}")

grid search took 197.9s
Best params: {'ccp_alpha': np.float64(3.6617010846511215e-05), 'class_weight': 'balanced', 'criterion': 'gini', 'min_samples_leaf': 1}
Best tuning macro-F1: 0.5509


In [ ]:
#CART forest hyperparameter tuning
# NBNBNB:
#no ccp_alpha -> RF trees are grown deep/unpruned on purpose (bagging controls variance)
#max_features -> subspace sampling size per split (tests the "insensitive to m" claim)

#Set flag true to perfrom tuning (takes like 40 minutes)
tune_flag = False

if tune_flag:
    param_grid = {
        "criterion": ["gini", "entropy"],
        "n_estimators": [100, 300],
        "max_features": ["sqrt", "log2", 0.3],
        "min_samples_leaf": [1, 5, 15],
        "class_weight": [None, "balanced"],
    }

    forest = RandomForestClassifier(random_state=42)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    grid_search_rf = GridSearchCV(
        estimator=forest,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
    )

    start = time.time()
    grid_search_rf.fit(X_tune, y_tune)
    print(f"grid search took {time.time() - start:.1f}s")

    print("Best params:", grid_search_rf.best_params_)
    print(f"Best tuning macro-F1: {grid_search_rf.best_score_:.4f}")


In [8]:
'''
/home/tiaan/miniconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
grid search took 2544.0s
Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_features': 0.3, 'min_samples_leaf': 5, 'n_estimators': 300}
Best tuning macro-F1: 0.5739
'''

"\n/home/tiaan/miniconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.\n  warnings.warn(\ngrid search took 2544.0s\nBest params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_features': 0.3, 'min_samples_leaf': 5, 'n_estimators': 300}\nBest tuning macro-F1: 0.5739\n"

## some findings from the tuning
- max_features=0.3 beat sqrt/log2. For your ~202 one-hot columns, sqrt ≈ 14 features and 0.3 ≈ 60. So the forest wanted many more features per split than the course's "insensitive to m ≈ √p" rule of thumb. That's a genuine finding — and the likely reason is exactly the one-hot fragmentation we predicted: with proto/service blown into ~190 sparse, mostly-zero dummy columns, a small random subset often contains nothing useful, so the forest needs a bigger subset per split. (This is precisely the advantage the C4.5-native arm should show — it samples from ~40 real features, proto as one.)
- class_weight='balanced' selected again — consistent with the A2 tree; the imbalance remedy keeps earning its place.
- min_samples_leaf=5 (mild leaf regularisation) and criterion='entropy'

In [23]:
#Now the CART forest needs to be induced on the full training set, using the optimal
#hyperparameter selection

#Set true to train (10 minutes)
train_flag = True

if train_flag:
    best_forest = RandomForestClassifier(
        n_estimators=300,
        criterion="entropy",
        max_features=0.3,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,          # parallelise the forest across cores
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    start = time.time()
    # one CV pass -> pooled out-of-fold predictions (RF parallel internally, folds sequential)
    y_pred = cross_val_predict(best_forest, X_train_test, y_train_test, cv=cv, n_jobs=1)
    print(f"cross_val_predict took {time.time() - start:.1f}s")

    # recover per-fold macro-F1 from the same (deterministic) folds
    y_arr = y_train_test.to_numpy()
    fold_scores = []
    for _, test_idx in cv.split(X_train_test, y_train_test):
        fold_scores.append(f1_score(y_arr[test_idx], y_pred[test_idx], average="macro"))
    fold_scores = np.array(fold_scores)
    print("Per-fold macro-F1:", fold_scores)
    print(f"Mean: {fold_scores.mean():.4f}  Std: {fold_scores.std():.4f}")

    print(classification_report(y_train_test, y_pred, target_names=list(attack_category_map.values())))
    print(confusion_matrix(y_train_test, y_pred))

cross_val_predict took 586.2s
Per-fold macro-F1: [0.59688038 0.58566778 0.59350455 0.57692426 0.58548571]
Mean: 0.5877  Std: 0.0070
                precision    recall  f1-score   support

        Normal       0.95      0.89      0.92     74950
Reconnaissance       0.88      0.80      0.84     11273
      Backdoor       0.03      0.19      0.06      1877
           DoS       0.33      0.40      0.36     13179
      Exploits       0.85      0.57      0.68     35868
      Analysis       0.05      0.14      0.07      2158
       Fuzzers       0.64      0.73      0.68     19538
         Worms       0.57      0.77      0.65       140
     Shellcode       0.49      0.86      0.62      1218
       Generic       1.00      0.98      0.99     47446

      accuracy                           0.79    207647
     macro avg       0.58      0.63      0.59    207647
  weighted avg       0.85      0.79      0.81    207647

[[66526    51     3    63   539    99  7434     2   229     4]
 [   16  8991   50

# C4.5 Tuning and training
Firstly we have to tune and train the regular tree.  


In [24]:
#Exporting the training and tuning data so it can be passed into the java implementation
df_C45 = pd.DataFrame(index=df.index)

#service - keep raw, missing as its own category. Note no on-hot
df_C45["service"] = df["service"].fillna("missing")

#proto -> keep raw categorical
df_C45["proto"] = df["proto"]
#raw categorical
df_C45["state"] = df["state"]
#raw categorical
df_C45["ct_state_ttl"] = df["ct_state_ttl"]

#Add the rest of the predictors (same numeric features as the CART pipeline, unchanged)
df_C45["dur"] = df["dur"]
df_C45["spkts"] = df["spkts"]
df_C45["dpkts"] = df["dpkts"]
df_C45["sbytes"] = df["sbytes"]
df_C45["dbytes"] = df["dbytes"]
df_C45["rate"] = df["rate"]
df_C45["sttl"] = df["sttl"]
df_C45["dttl"] = df["dttl"]
df_C45["sload"] = df["sload"]
df_C45["dload"] = df["dload"]
df_C45["sloss"] = df["sloss"]
df_C45["dloss"] = df["dloss"]
df_C45["sinpkt"] = df["sinpkt"]
df_C45["dinpkt"] = df["dinpkt"]
df_C45["sjit"] = df["sjit"]
df_C45["djit"] = df["djit"]
df_C45["swin"] = df["swin"]
df_C45["stcpb"] = df["stcpb"]
df_C45["dtcpb"] = df["dtcpb"]
df_C45["dwin"] = df["dwin"]
df_C45["tcprtt"] = df["tcprtt"]
df_C45["synack"] = df["synack"]
df_C45["ackdat"] = df["ackdat"]
df_C45["smean"] = df["smean"]
df_C45["dmean"] = df["dmean"]
df_C45["trans_depth"] = df["trans_depth"]
df_C45["response_body_len"] = df["response_body_len"]
df_C45["ct_srv_src"] = df["ct_srv_src"]
df_C45["ct_dst_ltm"] = df["ct_dst_ltm"]
df_C45["ct_src_dport_ltm"] = df["ct_src_dport_ltm"]
df_C45["ct_dst_sport_ltm"] = df["ct_dst_sport_ltm"]
df_C45["ct_dst_src_ltm"] = df["ct_dst_src_ltm"]
df_C45["is_ftp_login"] = df["is_ftp_login"]
df_C45["ct_ftp_cmd"] = df["ct_ftp_cmd"]
df_C45["ct_flw_http_mthd"] = df["ct_flw_http_mthd"]
df_C45["ct_src_ltm"] = df["ct_src_ltm"]
df_C45["ct_srv_dst"] = df["ct_srv_dst"]
df_C45["is_sm_ips_ports"] = df["is_sm_ips_ports"]

#target feature -> already integer-coded 0-9 in the raw data, passes straight through
df_C45["attack_cat"] = df["attack_cat"]

#remove observations where is_ftp_login has invalid values (same 26 rows as CART)
is_ftp_login_valid = (df["is_ftp_login"] == 0) | (df["is_ftp_login"] == 1)
df_C45 = df_C45.loc[is_ftp_login_valid]

#Split into tuning and training/testing set (same indices as CART) and export for the Java implementation
df_C45.loc[X_tune.index].to_csv("c45-java/tune.csv", index=False)
df_C45.loc[X_train_test.index].to_csv("c45-java/train_test.csv", index=False)

df_C45.shape


(257647, 43)

# C4.5 Tree tuning

In [ ]:
C45_train_flag = False

if C45_train_flag:
    C45_DIR = pathlib.Path("c45-java")   # relative to the notebook

    #compile the implementation
    compile_proc = subprocess.run(
        ["javac", "C45.java"],
        cwd=C45_DIR, capture_output=True, text=True,
    )
    if compile_proc.returncode != 0:
        print(compile_proc.stdout)
        print(compile_proc.stderr)
        raise RuntimeError("C45.java failed to compile")

    #tune the single C4.5 tree: stratified 5-fold macro-F1 grid search over
    #    criterion x min_samples_leaf x class_weight x prune  (32 configs, parallelised across cores).
    #    Reads c45-java/tune.csv and writes c45-java/best_standard.txt.
    start = time.time()
    proc = subprocess.Popen(
        ["java", "C45", "Tune", "Standard"],
        cwd=C45_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:          # stream the per-config lines as they finish
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError("C4.5 tree tuning failed")

    print(f"\nC4.5 tree tuning took {time.time() - start:.1f}s")


# C4.5 Tree training/testing

In [ ]:
# compile + run the Java eval -> reads best_standard.txt, runs 5-fold on
#train_test.csv, writes c45-java/predictions_standard.csv
subprocess.run(["javac", "C45.java"], cwd="c45-java", check=True)

start = time.time()
proc = subprocess.Popen(
    ["java", "C45", "Train_Test", "Standard"],
    cwd="c45-java", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError("C4.5 tree train/test failed")
print(f"C4.5 tree train/test took {time.time() - start:.1f}s")

# score the out-of-fold predictions against the true labels.
# row_id matches train_test.csv row order, which is X_train_test order -> y_train_test lines up.
preds = pd.read_csv("c45-java/predictions_standard.csv").sort_values("row_id")
y_pred = preds["prediction"].to_numpy()
y_true = y_train_test.to_numpy()

labels = list(range(10))
target_names = []
for i in labels:
    target_names.append(attack_category_map[i])

macro_f1 = f1_score(y_true, y_pred, average="macro", labels=labels)
print("\nC4.5 tree macro-F1 (train/test):", round(macro_f1, 4))
print(classification_report(y_true, y_pred, labels=labels, target_names=target_names, digits=3))


loaded train_test.csv: 207647 rows, 42 features, 10 classes; using 8 workers
wrote predictions_standard.csv (207647 rows)
C4.5 tree train/test took 15.3s

C4.5 tree macro-F1 (train/test): 0.5769
                precision    recall  f1-score   support

        Normal      0.915     0.915     0.915     74950
Reconnaissance      0.893     0.757     0.820     11273
      Backdoor      0.282     0.093     0.140      1877
           DoS      0.324     0.345     0.334     13179
      Exploits      0.643     0.741     0.688     35868
      Analysis      0.468     0.120     0.192      2158
       Fuzzers      0.634     0.588     0.611     19538
         Worms      0.493     0.479     0.486       140
     Shellcode      0.590     0.607     0.598      1218
       Generic      0.990     0.984     0.987     47446

      accuracy                          0.807    207647
     macro avg      0.623     0.563     0.577    207647
  weighted avg      0.808     0.807     0.805    207647



In [26]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
cm_df


,Normal,Reconnaissance,Backdoor,DoS,Exploits,Analysis,Fuzzers,Worms,Shellcode,Generic
Normal,68554,36,6,117,599,207,5280,3,108,40
Reconnaissance,48,8534,20,774,1843,2,15,6,25,6
Backdoor,3,16,175,571,955,6,130,2,12,7
DoS,107,86,57,4543,8015,21,208,5,63,74
Exploits,621,829,191,6440,26561,46,705,44,157,274
Analysis,180,1,34,566,994,260,123,0,0,0
Fuzzers,5299,23,121,710,1707,14,11498,4,130,32
Worms,2,3,6,7,48,0,4,67,0,3
Shellcode,90,21,6,66,160,0,116,0,739,20
Generic,33,5,5,243,411,0,50,5,19,46675


# C4.5 Forest tuning

In [13]:
C45_forest_flag = False
if C45_forest_flag:
    C45_DIR = pathlib.Path("c45-java")

    # 1) compile (cheap; safe to re-run)
    compile_proc = subprocess.run(
        ["javac", "C45.java"],
        cwd=C45_DIR, capture_output=True, text=True,
    )
    if compile_proc.returncode != 0:
        print(compile_proc.stdout)
        print(compile_proc.stderr)
        raise RuntimeError("C45.java failed to compile")

    # 2) tune the forest: out-of-bag macro-F1 grid search over
    #    criterion x max_features x min_samples_leaf x class_weight x n_estimators (96 configs).
    #    Reads c45-java/tune.csv and writes c45-java/best_forest.txt.
    start = time.time()
    proc = subprocess.Popen(
        ["java", "C45", "Tune", "Forest"],
        cwd=C45_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in proc.stdout:          # stream each config's OOB macro-F1 as it finishes
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError("C4.5 forest tuning failed")

    print(f"\nC4.5 forest tuning took {time.time() - start:.1f}s")


In [14]:
'''loaded tune.csv: 50000 rows, 42 features, 10 classes; using 8 workers
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=1 classWeight=none nTrees=100 -> oobMacroF1=0.5211
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=1 classWeight=none nTrees=300 -> oobMacroF1=0.5224
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=1 classWeight=balanced nTrees=100 -> oobMacroF1=0.5031
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=1 classWeight=balanced nTrees=300 -> oobMacroF1=0.5125
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=5 classWeight=none nTrees=100 -> oobMacroF1=0.4861
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=5 classWeight=none nTrees=300 -> oobMacroF1=0.4849
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=5 classWeight=balanced nTrees=100 -> oobMacroF1=0.5036
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=5 classWeight=balanced nTrees=300 -> oobMacroF1=0.5112
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=15 classWeight=none nTrees=100 -> oobMacroF1=0.4562
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=15 classWeight=none nTrees=300 -> oobMacroF1=0.4595
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=15 classWeight=balanced nTrees=100 -> oobMacroF1=0.4972
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=15 classWeight=balanced nTrees=300 -> oobMacroF1=0.4986
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=30 classWeight=none nTrees=100 -> oobMacroF1=0.4343
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=30 classWeight=none nTrees=300 -> oobMacroF1=0.4396
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=30 classWeight=balanced nTrees=100 -> oobMacroF1=0.4898
  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=30 classWeight=balanced nTrees=300 -> oobMacroF1=0.4910
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=1 classWeight=none nTrees=100 -> oobMacroF1=0.5510
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=1 classWeight=none nTrees=300 -> oobMacroF1=0.5487
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=1 classWeight=balanced nTrees=100 -> oobMacroF1=0.5273
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=1 classWeight=balanced nTrees=300 -> oobMacroF1=0.5300
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=5 classWeight=none nTrees=100 -> oobMacroF1=0.5294
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=5 classWeight=none nTrees=300 -> oobMacroF1=0.5320
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=5 classWeight=balanced nTrees=100 -> oobMacroF1=0.5613
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=5 classWeight=balanced nTrees=300 -> oobMacroF1=0.5528
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=15 classWeight=none nTrees=100 -> oobMacroF1=0.4996
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=15 classWeight=none nTrees=300 -> oobMacroF1=0.5016
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=15 classWeight=balanced nTrees=100 -> oobMacroF1=0.5420
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=15 classWeight=balanced nTrees=300 -> oobMacroF1=0.5425
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=30 classWeight=none nTrees=100 -> oobMacroF1=0.4765
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=30 classWeight=none nTrees=300 -> oobMacroF1=0.4791
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=30 classWeight=balanced nTrees=100 -> oobMacroF1=0.5160
  criterion=INFO_GAIN maxFeatures=13 minSamplesLeaf=30 classWeight=balanced nTrees=300 -> oobMacroF1=0.5199
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=1 classWeight=none nTrees=100 -> oobMacroF1=0.5642
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=1 classWeight=none nTrees=300 -> oobMacroF1=0.5632
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=1 classWeight=balanced nTrees=100 -> oobMacroF1=0.5314
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=1 classWeight=balanced nTrees=300 -> oobMacroF1=0.5344
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=5 classWeight=none nTrees=100 -> oobMacroF1=0.5329
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=5 classWeight=none nTrees=300 -> oobMacroF1=0.5451
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=5 classWeight=balanced nTrees=100 -> oobMacroF1=0.5640
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=5 classWeight=balanced nTrees=300 -> oobMacroF1=0.5659
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=15 classWeight=none nTrees=100 -> oobMacroF1=0.5117
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=15 classWeight=none nTrees=300 -> oobMacroF1=0.5110
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=15 classWeight=balanced nTrees=100 -> oobMacroF1=0.5490
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=15 classWeight=balanced nTrees=300 -> oobMacroF1=0.5508
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=30 classWeight=none nTrees=100 -> oobMacroF1=0.4884
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=30 classWeight=none nTrees=300 -> oobMacroF1=0.4882
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=30 classWeight=balanced nTrees=100 -> oobMacroF1=0.5308
  criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=30 classWeight=balanced nTrees=300 -> oobMacroF1=0.5278
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=1 classWeight=none nTrees=100 -> oobMacroF1=0.5212
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=1 classWeight=none nTrees=300 -> oobMacroF1=0.5247
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=1 classWeight=balanced nTrees=100 -> oobMacroF1=0.4878
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=1 classWeight=balanced nTrees=300 -> oobMacroF1=0.4933
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=5 classWeight=none nTrees=100 -> oobMacroF1=0.4842
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=5 classWeight=none nTrees=300 -> oobMacroF1=0.4892
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=5 classWeight=balanced nTrees=100 -> oobMacroF1=0.5095
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=5 classWeight=balanced nTrees=300 -> oobMacroF1=0.5040
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=15 classWeight=none nTrees=100 -> oobMacroF1=0.4680
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=15 classWeight=none nTrees=300 -> oobMacroF1=0.4707
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=15 classWeight=balanced nTrees=100 -> oobMacroF1=0.4890
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=15 classWeight=balanced nTrees=300 -> oobMacroF1=0.4916
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=30 classWeight=none nTrees=100 -> oobMacroF1=0.4399
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=30 classWeight=none nTrees=300 -> oobMacroF1=0.4421
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=30 classWeight=balanced nTrees=100 -> oobMacroF1=0.4834
  criterion=GAIN_RATIO maxFeatures=6 minSamplesLeaf=30 classWeight=balanced nTrees=300 -> oobMacroF1=0.4887
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=1 classWeight=none nTrees=100 -> oobMacroF1=0.5416
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=1 classWeight=none nTrees=300 -> oobMacroF1=0.5442
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=1 classWeight=balanced nTrees=100 -> oobMacroF1=0.5415
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=1 classWeight=balanced nTrees=300 -> oobMacroF1=0.5385
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=5 classWeight=none nTrees=100 -> oobMacroF1=0.5187
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=5 classWeight=none nTrees=300 -> oobMacroF1=0.5218
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=5 classWeight=balanced nTrees=100 -> oobMacroF1=0.5479
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=5 classWeight=balanced nTrees=300 -> oobMacroF1=0.5552
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=15 classWeight=none nTrees=100 -> oobMacroF1=0.4986
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=15 classWeight=none nTrees=300 -> oobMacroF1=0.4988
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=15 classWeight=balanced nTrees=100 -> oobMacroF1=0.5319
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=15 classWeight=balanced nTrees=300 -> oobMacroF1=0.5341
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=30 classWeight=none nTrees=100 -> oobMacroF1=0.4796
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=30 classWeight=none nTrees=300 -> oobMacroF1=0.4756
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=30 classWeight=balanced nTrees=100 -> oobMacroF1=0.5146
  criterion=GAIN_RATIO maxFeatures=13 minSamplesLeaf=30 classWeight=balanced nTrees=300 -> oobMacroF1=0.5163
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=1 classWeight=none nTrees=100 -> oobMacroF1=0.5354
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=1 classWeight=none nTrees=300 -> oobMacroF1=0.5294
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=1 classWeight=balanced nTrees=100 -> oobMacroF1=0.5359
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=1 classWeight=balanced nTrees=300 -> oobMacroF1=0.5379
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=5 classWeight=none nTrees=100 -> oobMacroF1=0.5213
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=5 classWeight=none nTrees=300 -> oobMacroF1=0.5251
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=5 classWeight=balanced nTrees=100 -> oobMacroF1=0.5461
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=5 classWeight=balanced nTrees=300 -> oobMacroF1=0.5530
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=15 classWeight=none nTrees=100 -> oobMacroF1=0.5068
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=15 classWeight=none nTrees=300 -> oobMacroF1=0.5096
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=15 classWeight=balanced nTrees=100 -> oobMacroF1=0.5331
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=15 classWeight=balanced nTrees=300 -> oobMacroF1=0.5339
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=30 classWeight=none nTrees=100 -> oobMacroF1=0.4828
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=30 classWeight=none nTrees=300 -> oobMacroF1=0.4835
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=30 classWeight=balanced nTrees=100 -> oobMacroF1=0.5177
  criterion=GAIN_RATIO maxFeatures=21 minSamplesLeaf=30 classWeight=balanced nTrees=300 -> oobMacroF1=0.5179
best forest: criterion=INFO_GAIN maxFeatures=21 minSamplesLeaf=5 classWeight=balanced nTrees=300 oobMacroF1=0.5659

C4.5 forest tuning took 14405.8s
'''

'loaded tune.csv: 50000 rows, 42 features, 10 classes; using 8 workers\n  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=1 classWeight=none nTrees=100 -> oobMacroF1=0.5211\n  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=1 classWeight=none nTrees=300 -> oobMacroF1=0.5224\n  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=1 classWeight=balanced nTrees=100 -> oobMacroF1=0.5031\n  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=1 classWeight=balanced nTrees=300 -> oobMacroF1=0.5125\n  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=5 classWeight=none nTrees=100 -> oobMacroF1=0.4861\n  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=5 classWeight=none nTrees=300 -> oobMacroF1=0.4849\n  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=5 classWeight=balanced nTrees=100 -> oobMacroF1=0.5036\n  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=5 classWeight=balanced nTrees=300 -> oobMacroF1=0.5112\n  criterion=INFO_GAIN maxFeatures=6 minSamplesLeaf=15 classWeight=none nTrees=100 -> oob

# Re-tuning the forests with optimal hyperparameters + more trees
Normally we'd want to because more trees always reduces error until a plateau, but by the time we reach 300 trees we've already plateaud. In the case of the C4.5 forest if you look at any config: 100→300 moves the score by only ~0.001–0.002, and sometimes down (e.g. mf=21/leaf=1/none: 100→0.5642, 300→0.5632). That's pure OOB noise, not improvement.  
  
For the CART forest, we only need to test if it had already plateaud going from 100 trees to 300 trees:

  


In [15]:
check_plateau = False

if check_plateau:
    # fix the tuned optimum, vary only n_estimators
    plateau_grid = {"n_estimators": [100, 300]}

    forest = RandomForestClassifier(
        criterion="entropy",
        max_features=0.3,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    plateau_search = GridSearchCV(
        estimator=forest,
        param_grid=plateau_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
    )

    start = time.time()
    plateau_search.fit(X_tune, y_tune)
    print(f"took {time.time() - start:.1f}s")

    res = pd.DataFrame(plateau_search.cv_results_)
    print(res[["param_n_estimators", "mean_test_score", "std_test_score"]]
        .sort_values("param_n_estimators").to_string(index=False))


**It has plateaud:** The jump from 100→300 is only +0.0053 in macro f1 score

# C4.5 final forest training/testing

In [ ]:
# compile + run the Java eval -> reads best_forest.txt runs stratified 5-fold on
# train_test.csv, writes predictions_forest.csv
subprocess.run(["javac", "C45.java"], cwd="c45-java", check=True)

start = time.time()
proc = subprocess.Popen(
    ["java", "C45", "Train_Test", "Forest"],
    cwd="c45-java", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError("C4.5 forest train/test failed")
print(f"C4.5 forest train/test took {time.time() - start:.1f}s")

#score the out-of-fold predictions against the true labels
#(row_id matches train_test.csv order, which is X_train_test order -> y_train_test lines up)
preds = pd.read_csv("c45-java/predictions_forest.csv").sort_values("row_id")
y_pred = preds["prediction"].to_numpy()
y_true = y_train_test.to_numpy()

labels = list(range(10))
target_names = []
for i in labels:
    target_names.append(attack_category_map[i])

macro_f1 = f1_score(y_true, y_pred, average="macro", labels=labels)
print("\nC4.5 forest macro-F1 (train/test):", round(macro_f1, 4))
print(classification_report(y_true, y_pred, labels=labels, target_names=target_names, digits=3))


loaded train_test.csv: 207647 rows, 42 features, 10 classes; using 8 workers
wrote predictions_forest.csv (207647 rows)
C4.5 forest train/test took 3022.1s

C4.5 forest macro-F1 (train/test): 0.5855
                precision    recall  f1-score   support

        Normal      0.950     0.899     0.924     74950
Reconnaissance      0.882     0.797     0.837     11273
      Backdoor      0.025     0.206     0.045      1877
           DoS      0.353     0.249     0.292     13179
      Exploits      0.855     0.568     0.683     35868
      Analysis      0.035     0.133     0.055      2158
       Fuzzers      0.653     0.715     0.683     19538
         Worms      0.641     0.764     0.697       140
     Shellcode      0.535     0.828     0.650      1218
       Generic      0.998     0.980     0.989     47446

      accuracy                          0.781    207647
     macro avg      0.593     0.614     0.585    207647
  weighted avg      0.855     0.781     0.811    207647



In [17]:
'''
loaded train_test.csv: 207647 rows, 42 features, 10 classes; using 8 workers
wrote predictions_forest.csv (207647 rows)
C4.5 forest train/test took 5197.3s

C4.5 forest macro-F1 (train/test): 0.5855
                precision    recall  f1-score   support

        Normal      0.950     0.899     0.924     74950
Reconnaissance      0.882     0.797     0.837     11273
      Backdoor      0.025     0.206     0.045      1877
           DoS      0.353     0.249     0.292     13179
      Exploits      0.855     0.568     0.683     35868
      Analysis      0.035     0.133     0.055      2158
       Fuzzers      0.653     0.715     0.683     19538
         Worms      0.641     0.764     0.697       140
     Shellcode      0.535     0.828     0.650      1218
       Generic      0.998     0.980     0.989     47446

      accuracy                          0.781    207647
     macro avg      0.593     0.614     0.585    207647
  weighted avg      0.855     0.781     0.811    207647

'''

'\nloaded train_test.csv: 207647 rows, 42 features, 10 classes; using 8 workers\nwrote predictions_forest.csv (207647 rows)\nC4.5 forest train/test took 5197.3s\n\nC4.5 forest macro-F1 (train/test): 0.5855\n                precision    recall  f1-score   support\n\n        Normal      0.950     0.899     0.924     74950\nReconnaissance      0.882     0.797     0.837     11273\n      Backdoor      0.025     0.206     0.045      1877\n           DoS      0.353     0.249     0.292     13179\n      Exploits      0.855     0.568     0.683     35868\n      Analysis      0.035     0.133     0.055      2158\n       Fuzzers      0.653     0.715     0.683     19538\n         Worms      0.641     0.764     0.697       140\n     Shellcode      0.535     0.828     0.650      1218\n       Generic      0.998     0.980     0.989     47446\n\n      accuracy                          0.781    207647\n     macro avg      0.593     0.614     0.585    207647\n  weighted avg      0.855     0.781     0.811   

In [ ]:
preds  = pd.read_csv("c45-java/predictions_forest.csv").sort_values("row_id")
y_true = y_train_test.to_numpy()
y_pred = preds["prediction"].to_numpy()
labels = list(range(10))

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print(cm_df)


                Normal  Reconnaissance  Backdoor   DoS  Exploits  Analysis  \
Normal           67372              52         2    53       562        87   
Reconnaissance      13            8987       872   502       466       368   
Backdoor             0              11       386   585        69       774   
DoS                 38              95      5190  3277      1579      2685   
Exploits           320             991      6382  3648     20390      3386   
Analysis           212               2       943   643        64       286   
Fuzzers           2930              29      1249   409       174       584   
Worms                0               0         0     4        23         0   
Shellcode           30              16         0     6        55         0   
Generic             12               6       154   166       464        45   

                Fuzzers  Worms  Shellcode  Generic  
Normal             6645      2        168        7  
Reconnaissance       21      2     

In [19]:
cm_df

,Normal,Reconnaissance,Backdoor,DoS,Exploits,Analysis,Fuzzers,Worms,Shellcode,Generic
Normal,67372,52,2,53,562,87,6645,2,168,7
Reconnaissance,13,8987,872,502,466,368,21,2,41,1
Backdoor,0,11,386,585,69,774,29,2,20,1
DoS,38,95,5190,3277,1579,2685,129,4,172,10
Exploits,320,991,6382,3648,20390,3386,400,37,266,48
Analysis,212,2,943,643,64,286,8,0,0,0
Fuzzers,2930,29,1249,409,174,584,13977,5,169,12
Worms,0,0,0,4,23,0,6,107,0,0
Shellcode,30,16,0,6,55,0,103,0,1008,0
Generic,12,6,154,166,464,45,73,8,40,46478
